In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data_etna"

sys.path.append(str(SRC_DIR))

from cause_trigger import (
    CauseTriggerConfig,
    run_cause_trigger,
    find_increase_split,
)

from parameter_extraction import find_parameters

In [ ]:
etna_path = DATA_DIR / "FINAL_ME01_scaled.csv"

etna = pd.read_csv(etna_path, parse_dates=["time"])

print(etna.shape)
display(etna.head())
display(etna.isna().mean().sort_values())

In [ ]:
X_all = (
    etna
    .drop(columns=["station"], errors="ignore")
    .set_index("time")
    .sort_index()
)

X_all = X_all.select_dtypes(include=[np.number])

print(X_all.shape)
display(X_all.head())
display(X_all.isna().sum())

In [ ]:
etna_event_time = pd.Timestamp("2008-05-12 06:28:00", tz="UTC")

start = pd.Timestamp("2008-05-12 00:00:00", tz="UTC")
end = pd.Timestamp("2008-05-13 12:00:00", tz="UTC")

X_event = X_all.loc[start:end].copy()

print(X_event.shape)
print(X_event.index.min(), "to", X_event.index.max())
display(X_event.head())

In [ ]:
def split_diagnostics(X, target, event_time=None, min_interval_length=30):
    split_idx = find_increase_split(
        X[target],
        min_interval_length=min_interval_length,
    )

    if split_idx is None:
        return {
            "target": target,
            "split_index": None,
            "split_time": None,
            "I1_length": None,
            "I2_length": None,
            "abs_mean_I1": None,
            "abs_mean_I2": None,
            "abs_mean_difference": None,
            "distance_to_event": None,
        }

    I1 = X.iloc[:split_idx]
    I2 = X.iloc[split_idx:]

    split_time = X.index[split_idx]

    out = {
        "target": target,
        "split_index": split_idx,
        "split_time": split_time,
        "I1_length": len(I1),
        "I2_length": len(I2),
        "abs_mean_I1": abs(I1[target].mean()),
        "abs_mean_I2": abs(I2[target].mean()),
        "abs_mean_difference": abs(I2[target].mean()) - abs(I1[target].mean()),
        "distance_to_event": None,
    }

    if event_time is not None:
        out["distance_to_event"] = split_time - event_time

    return out


candidate_targets = [
    "T_log_scaled",   # known teleseismic trigger band; good sanity check
    "Y_log_scaled",   # high-frequency response/effect band
    "S_log_scaled",   # background/state band
]

split_table = pd.DataFrame([
    split_diagnostics(
        X_event,
        target=t,
        event_time=etna_event_time,
        min_interval_length=30,
    )
    for t in candidate_targets
])

display(split_table)

In [ ]:
def summarize_result(result):
    print("Backend:", result.get("backend"))
    print("Split index:", result.get("split_index"))
    print("B2:", result.get("B_2"))
    print("Trigger candidates:", result.get("T_candidates"))
    print("Accepted triggers:", result.get("T"))
    print("Causes:", result.get("C"))
    print("Pairs:", result.get("pairs"))

    diag = pd.DataFrame(result.get("diagnostics", []))

    if len(diag) > 0:
        sort_cols = [c for c in ["accepted", "rss_reduction_ratio", "f_stat"] if c in diag.columns]

        if "rss_reduction_ratio" in diag.columns:
            diag = diag.sort_values("rss_reduction_ratio", ascending=False)

        display(diag)

    return diag

In [ ]:
config_pcmci_T = CauseTriggerConfig(
    y_t="T_log_scaled",
    lags=3,
    causal_backend="pcmci",
    alpha=0.05,
    min_interval_length=30,
    beta_is_ones=True,

    # PCMCI settings
    pcmci_pc_alpha=0.05,
    pcmci_alpha_level=0.05,
    pcmci_fdr_method="fdr_bh",
    pcmci_cond_ind_test="parcorr",
    pcmci_verbosity=0,
)

result_pcmci_T = run_cause_trigger(X_event, config_pcmci_T)
diag_pcmci_T = summarize_result(result_pcmci_T)

In [ ]:
result_pcmci_T.get("causal_lags")

In [ ]:
result_pcmci_T.get("causal_scores")

In [ ]:
config_pcmci_Y = CauseTriggerConfig(
    y_t="Y_log_scaled",
    lags=3,
    causal_backend="pcmci",
    alpha=0.05,
    min_interval_length=30,
    beta_is_ones=True,

    pcmci_pc_alpha=0.05,
    pcmci_alpha_level=0.05,
    pcmci_fdr_method="fdr_bh",
    pcmci_cond_ind_test="parcorr",
    pcmci_verbosity=0,
)

result_pcmci_Y = run_cause_trigger(X_event, config_pcmci_Y)
diag_pcmci_Y = summarize_result(result_pcmci_Y)

In [ ]:
config_hmml_T = CauseTriggerConfig(
    y_t="T_log_scaled",
    lags=3,
    distribution="gaussian",
    causal_backend="hmml",
    alpha=0.05,
    min_interval_length=30,
    beta_is_ones=True,
)

result_hmml_T = run_cause_trigger(X_event, config_hmml_T)
diag_hmml_T = summarize_result(result_hmml_T)

In [ ]:
target = "T_log_scaled"

distribution, lag = find_parameters(
    X=df_model,
    target_series=df_model[target],
    max_lags=3,
    criterion="aic",
    fallback_lag=1,
    fallback_distribution="gaussian",
)

config = CauseTriggerConfig(
    y_t=target,
    lags=lag,
    distribution=distribution,
    causal_backend="hmml",
    min_I1_length=12,
    min_I2_length=24,
)

config.parameter_source = "VAR_AIC_and_distfit"

In [ ]:
print("Selected target:", target)
print("Selected lag:", lag)
print("Selected distribution:", distribution)

In [ ]:
def compact_result_row(name, target, result):
    return {
        "run": name,
        "target": target,
        "backend": result.get("backend"),
        "split_index": result.get("split_index"),
        "B2": result.get("B_2"),
        "T_candidates": result.get("T_candidates"),
        "accepted_triggers": result.get("T"),
        "causes": result.get("C"),
        "pairs": result.get("pairs"),
    }


comparison = pd.DataFrame([
    compact_result_row("pcmci_T", "T_log_scaled", result_pcmci_T),
    compact_result_row("pcmci_Y", "Y_log_scaled", result_pcmci_Y),
    compact_result_row("hmml_T", "T_log_scaled", result_hmml_T),
    compact_result_row("hmml_Y", "Y_log_scaled", result_hmml_Y),
])

display(comparison)

In [ ]:
def add_run_label(diag, run_name, target, backend):
    if diag is None or len(diag) == 0:
        return pd.DataFrame()

    out = diag.copy()
    out["run"] = run_name
    out["target"] = target
    out["backend"] = backend
    return out


all_diag = pd.concat(
    [
        add_run_label(diag_pcmci_T, "pcmci_T", "T_log_scaled", "pcmci"),
        add_run_label(diag_pcmci_Y, "pcmci_Y", "Y_log_scaled", "pcmci"),
        add_run_label(diag_hmml_T, "hmml_T", "T_log_scaled", "hmml"),
        add_run_label(diag_hmml_Y, "hmml_Y", "Y_log_scaled", "hmml"),
    ],
    ignore_index=True,
)

display(
    all_diag.sort_values(
        ["accepted", "rss_reduction_ratio"],
        ascending=[False, False],
    )
)

In [ ]:
import matplotlib.pyplot as plt

plot_df = all_diag.dropna(subset=["rss_reduction_ratio"]).copy()

if len(plot_df) > 0:
    plot_df["label"] = (
        plot_df["backend"].astype(str)
        + " | "
        + plot_df["target"].astype(str)
        + " | "
        + plot_df["trigger"].astype(str)
    )

    plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
    plt.barh(plot_df["label"], plot_df["rss_reduction_ratio"])
    plt.xlabel("Relative RSS reduction")
    plt.title("Trigger effect size: Eq. (3) full model vs Eq. (4) reduced model")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
def run_sensitivity_grid(
    df_model,
    targets,
    backends=("hmml", "pcmci"),
    lags=(1, 2, 3),
    distributions=("gaussian",),
    min_I1_length=12,
    min_I2_length=24,
):
    rows = []

    for target in targets:
        for backend in backends:
            for lag in lags:
                for distribution in distributions:
                    config = CauseTriggerConfig(
                        y_t=target,
                        lags=lag,
                        distribution=distribution,
                        causal_backend=backend,
                        min_I1_length=min_I1_length,
                        min_I2_length=min_I2_length,
                        beta_is_ones=True,
                    )

                    try:
                        result = run_cause_trigger(df_model, config)
                        diagnostics = diagnostics_to_dataframe(result)

                        rows.append({
                            "target": target,
                            "backend": backend,
                            "lag": lag,
                            "distribution": distribution,
                            "split_timestamp": result.get("split_timestamp"),
                            "I1_length": result.get("I1_length"),
                            "I2_length": result.get("I2_length"),
                            "B_2": result.get("B_2"),
                            "B_2_non_target": result.get("B_2_non_target"),
                            "T_candidates": result.get("T_candidates"),
                            "accepted_triggers": result.get("T"),
                            "causes": result.get("C"),
                            "pairs": result.get("pairs"),
                            "n_diagnostics": len(diagnostics),
                        })

                    except Exception as e:
                        rows.append({
                            "target": target,
                            "backend": backend,
                            "lag": lag,
                            "distribution": distribution,
                            "error": str(e),
                        })

    return pd.DataFrame(rows)

In [ ]:
sensitivity_df = run_sensitivity_grid(
    df_model=df_model,
    targets=["T_log_scaled", "Y_log_scaled", "S_log_scaled"],
    backends=("hmml", "pcmci"),
    lags=(1, 2, 3),
    distributions=("gaussian",),
    min_I1_length=12,
    min_I2_length=24,
)

sensitivity_df

In [ ]:
import matplotlib.pyplot as plt

def plot_target_with_split(df, target, result, event_time=None):
    plt.figure(figsize=(12, 4))
    plt.plot(df.index, df[target], label=target)

    split_timestamp = result.get("split_timestamp")
    if split_timestamp is not None:
        plt.axvline(split_timestamp, linestyle="--", label="Detected split")

    if event_time is not None:
        plt.axvline(pd.Timestamp(event_time), linestyle=":", label="Known event time")

    plt.title(f"{target}: detected split vs known event")
    plt.xlabel("Time")
    plt.ylabel("Scaled value")
    plt.legend()
    plt.show()

In [ ]:
plot_target_with_split(
    df=event_window,
    target="T_log_scaled",
    result=event_result,
    event_time="2008-05-12 06:28:00",
)